In [1]:
# ── Cell 1: Mount Drive (run this first, every session) ──────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/mbert-uqa-improved'
import os; os.makedirs(DRIVE_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
# ============================================================
# mBERT on UQA  —  IMPROVED VERSION  (Assignment 3)
# Changes vs A2:
#   1. Sliding window tokenisation (max_length=384, stride=128)
#   2. Correct Q/C tokeniser call (separate args, not merged string)
#   3. Epochs 3 → 6, batch 4 → 8, grad_accum 2 → 4
#   4. weight_decay=0.01 added
#   5. eval_steps 10000 → 500  (visible training curve)
#   6. save_total_limit 2 → 3
#   7. seed=42 for reproducibility
#   8. Eval: max_length=384 matches training (was 512)
#   9. Eval: end >= start guard added
#  10. Batched evaluation (16x faster than sample-by-sample)
# ============================================================
!pip install transformers datasets accelerate evaluate sentencepiece -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [3]:
MODEL   = "bert-base-multilingual-cased"
REPO    = "mbert-uqa-improved"
EPOCHS  =   6    # was 6
MAX_LEN = 384     # was 512
STRIDE  = 128     # NEW — sliding window
SEED    = 42      # NEW — reproducibility
LR      = 2e-5

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 1. Load Dataset

In [5]:
from datasets import load_dataset

def filter_answerable(example):
    return not example['is_impossible']

raw = load_dataset("uqa/UQA")
raw["train"]      = raw["train"].filter(filter_answerable)
raw["validation"] = raw["validation"].filter(filter_answerable)
print(raw)
# Train: 83,018 | Validation: 11,169

README.md:   0%|          | 0.00/898 [00:00<?, ?B/s]

data/train-00000-of-00001-bac007e8ca7192(…):   0%|          | 0.00/30.2M [00:00<?, ?B/s]

data/validation-00000-of-00001-cf8a6960d(…):   0%|          | 0.00/2.92M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/124745 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/16824 [00:00<?, ? examples/s]

Filter:   0%|          | 0/124745 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16824 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'],
        num_rows: 83018
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'],
        num_rows: 11169
    })
})


## 2. Preprocessing  (sliding window — FIX #1 & #2)

In [6]:
# FIX #1: sliding window so long contexts are NOT truncated
# FIX #2: question & context passed as SEPARATE args (not merged string)

def preprocess(examples):
    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LEN,          # was 512
        truncation="only_second",    # only truncate context, never question
        stride=STRIDE,               # NEW: overlap windows
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map     = inputs.pop("overflow_to_sample_mapping")
    answers        = examples["answer"]
    answer_starts  = examples["answer_start"]

    start_positions, end_positions = [], []

    for i, offsets in enumerate(offset_mapping):
        sample_idx  = sample_map[i]
        answer      = answers[sample_idx]
        start_char  = answer_starts[sample_idx]
        end_char    = start_char + len(answer)
        sequence_ids = inputs.sequence_ids(i)

        # find context token range
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        ctx_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        ctx_end = idx - 1

        # if answer is outside this window, label (0,0)
        if offsets[ctx_start][0] > start_char or offsets[ctx_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # find exact token positions
            idx = ctx_start
            while idx <= ctx_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = ctx_end
            while idx >= ctx_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"]   = end_positions
    return inputs

In [7]:
train_dataset = raw["train"].map(
    preprocess, batched=True,
    remove_columns=raw["train"].column_names
)
val_dataset = raw["validation"].map(
    preprocess, batched=True,
    remove_columns=raw["validation"].column_names
)

print(f"Train windows : {len(train_dataset):,}")
print(f"Val windows   : {len(val_dataset):,}")
# Expect train > 83k because sliding window creates multiple windows per long context

Map:   0%|          | 0/83018 [00:00<?, ? examples/s]

Map:   0%|          | 0/11169 [00:00<?, ? examples/s]

Train windows : 96,467
Val windows   : 13,489


## 3. Model

In [8]:
from transformers import AutoModelForQuestionAnswering
model = AutoModelForQuestionAnswering.from_pretrained(MODEL)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

Parameters: 177,264,386


## 4. Training  (FIX #3 #4 #5 #6 #7)

In [9]:
# ── Cell X: HuggingFace Authentication (Optional) ────────────────────────
# Uncomment and run if pushing to hub is enabled below
# from huggingface_hub import login
# login(token="your_hf_token")   # or use Colab Secrets (recommended)

In [10]:
from transformers import TrainingArguments, Trainer, default_data_collator

training_args = TrainingArguments(
    output_dir=DRIVE_DIR,               # ← checkpoints go to Drive, not /content
    seed=SEED,                          # FIX #7 — reproducibility

    num_train_epochs=EPOCHS,            # FIX #3 — 6 (was 3)
    learning_rate=LR,
    weight_decay=0.01,                  # FIX #4 — was missing

    per_device_train_batch_size=8,      # FIX #3 — was 4
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,      # FIX #3 — was 2
    # effective batch = 8 × 4 = 32 per GPU

    fp16=True, # Set to False to disable mixed-precision training and avoid scaler errors if not needed.
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    eval_strategy="steps",
    eval_steps=500,                     # FIX #5 — was 10000 (blind!)
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,                 # keep only last 2 to save Drive space

    load_best_model_at_end=True,
    metric_for_best_model="loss",

    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator,
    processing_class=tokenizer,
)

In [13]:
from transformers.trainer_utils import get_last_checkpoint

last_ckpt = get_last_checkpoint(DRIVE_DIR)

if last_ckpt:
    print(f"Resuming from: {last_ckpt}")
else:
    print("No checkpoint found — starting fresh")

trainer.train(resume_from_checkpoint=last_ckpt)

Resuming from: /content/drive/MyDrive/mbert-uqa-improved/checkpoint-16000


There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Step,Training Loss,Validation Loss
16500,0.506966,1.884359


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

The Training was stopped becoz Model was not Learning anymore.

# Loading Best CheckPoint as Model was Overfittingloss_chart.svg

In [15]:
#Load the saved best model ────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

SAVE_PATH = '/content/drive/MyDrive/mbert-uqa-best-recovered'

tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
model = AutoModelForQuestionAnswering.from_pretrained(SAVE_PATH).to("cuda")
model.eval()
print("Model loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded successfully


## 5. Evaluation  (FIX #8 #9 #10)

In [16]:
#  validation set with merged duplicate IDs ─────────────────
from datasets import Dataset, load_dataset

def filter_answerable(example):
    return not example['is_impossible']

def merge_duplicate_ids(ds):
    data = ds.to_dict()
    grouped = {}
    for i in range(len(data['id'])):
        idx = data['id'][i]
        if idx not in grouped:
            grouped[idx] = {
                'id': idx,
                'title': data['title'][i],
                'context': data['context'][i],
                'question': data['question'][i],
                'is_impossible': data['is_impossible'][i],
                'answer': [data['answer'][i]],
                'answer_start': [data['answer_start'][i]],
            }
        else:
            grouped[idx]['answer'].append(data['answer'][i])
            grouped[idx]['answer_start'].append(data['answer_start'][i])
    return Dataset.from_dict(
        {k: [d[k] for d in grouped.values()] for k in list(grouped.values())[0]}
    )

raw = load_dataset("uqa/UQA")
raw["validation"] = raw["validation"].filter(filter_answerable)
eval_ds = merge_duplicate_ids(raw["validation"])
print(f"Evaluation examples: {len(eval_ds):,}")

Evaluation examples: 5,811


In [18]:
import evaluate
from tqdm import tqdm

squad_metric = evaluate.load("squad")
MAX_LEN = 384
BATCH_SIZE = 16

predictions, references = [], []

for start in tqdm(range(0, len(eval_ds), BATCH_SIZE)):
    batch = eval_ds[start : start + BATCH_SIZE]

    enc = tokenizer(
        batch["question"],
        batch["context"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        out = model(**enc)

    for i in range(len(batch["id"])):
        s = torch.argmax(out.start_logits[i]).item()
        e = torch.argmax(out.end_logits[i]).item()
        if e < s:
            e = s   # guard: end must be >= start

        pred_text = tokenizer.decode(
            enc["input_ids"][i][s : e + 1],
            skip_special_tokens=True
        )
        predictions.append({"id": batch["id"][i], "prediction_text": pred_text})
        references.append({
            "id": batch["id"][i],
            "answers": {
                "text": batch["answer"][i],
                "answer_start": batch["answer_start"][i],
            }
        })

results = squad_metric.compute(predictions=predictions, references=references)
print(f"\n{'='*40}")
print(f"  Exact Match (EM) : {results['exact_match']:.4f}")
print(f"  F1 Score         : {results['f1']:.4f}")
print(f"{'='*40}")
print(f"\nBaseline comparison:")
print(f"  mBERT A2 (bad config)  →  EM: 50.28  F1: 63.52")
print(f"  mBERT A3 (this run)    →  EM: {results['exact_match']:.2f}  F1: {results['f1']:.2f}")
print(f"  XLM-R A2 (reference)   →  EM: 64.45  F1: 77.14")

100%|██████████| 364/364 [02:48<00:00,  2.16it/s]



  Exact Match (EM) : 56.8061
  F1 Score         : 70.0031

Baseline comparison:
  mBERT A2 (bad config)  →  EM: 50.28  F1: 63.52
  mBERT A3 (this run)    →  EM: 56.81  F1: 70.00
  XLM-R A2 (reference)   →  EM: 64.45  F1: 77.14


## 6. Save Best Checkpoint to Google Drive

In [ ]:
# Save the best checkpoint as a zip to Google Drive for persistence
import shutil

best_ckpt = sorted([d for d in os.listdir(DRIVE_DIR) if d.startswith("checkpoint")])[-1]
ckpt_path = os.path.join(DRIVE_DIR, best_ckpt)
print(f"Best checkpoint: {ckpt_path}")
zip_dest  = "/content/drive/MyDrive/mbert_uqa_improved_best"

# Mount drive first if not already mounted:
# from google.colab import drive
# drive.mount('/content/drive')

shutil.make_archive(zip_dest, "zip", ckpt_path)
print(f"Saved: {zip_dest}.zip  from  {ckpt_path}")

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/mbert_uqa_improved_best.zip')